# Phase 3, step 4 — knowing when to stop

The step-3 notebook ended with a live run that worked. It also ended with a
problem, visible in its own leaderboard: **eleven experiments, and the last six
moved the score by about 0.003.**

Nothing was broken. The agent had tools, the tools worked, every experiment was
recorded. It simply had no notion of *done*, so it kept going — and would have
kept going until the context ran out, spending real money to re-roll a die.

This notebook is about handing that judgement to the harness. It is a smaller
slice than the previous ones, and the interesting part is not the code. It is
the two questions underneath:

1. What counts as an improvement, when every number is noisy?
2. Who gets to say the run is over — the model, or the harness?

## 1. The evidence

Here is the real leaderboard from the step-3 run, typed out. Look at it the way
you would look at someone else's experiment log.

In [ ]:
import json
import pandas as pd

observed = [
    {"id": "e1",  "model_type": "baseline", "cv_score": 0.6903, "fold_spread": 0.0004},
    {"id": "e2",  "model_type": "lgbm",     "cv_score": 0.5049, "fold_spread": 0.0374},
    {"id": "e3",  "model_type": "xgb",      "cv_score": 0.3694, "fold_spread": 0.0301},
    {"id": "e4",  "model_type": "catboost", "cv_score": 0.2744, "fold_spread": 0.0209},
    {"id": "e5",  "model_type": "catboost", "cv_score": 0.2779, "fold_spread": 0.0221},
    {"id": "e6",  "model_type": "catboost", "cv_score": 0.2987, "fold_spread": 0.0244},
    {"id": "e7",  "model_type": "catboost", "cv_score": 0.2955, "fold_spread": 0.0233},
    {"id": "e8",  "model_type": "xgb",      "cv_score": 0.2741, "fold_spread": 0.0252},
    {"id": "e9",  "model_type": "xgb",      "cv_score": 0.2718, "fold_spread": 0.0247},
    {"id": "e10", "model_type": "lgbm",     "cv_score": 0.3130, "fold_spread": 0.0288},
    {"id": "e11", "model_type": "catboost", "cv_score": 0.2779, "fold_spread": 0.0231},
]
pd.DataFrame(observed)

The run really got somewhere: `0.6903` to `0.2744` in four experiments. Then
seven more experiments bought `0.2744` to `0.2718` — a gain of `0.0026`, while
each model's own score wobbles by `±0.02` between folds.

That last comparison is the whole notebook. The improvement is **an order of
magnitude smaller than the noise in the measurement it is made of**. If you
re-ran e9 with a different fold seed it could easily come out worse than e4.

**Check yourself:** e9 really does have the lowest number. What is wrong with
shipping it?

> Nothing, exactly — it is a fine model, and `finalize_run` will happily package
> it. What is wrong is *believing* it is better than e4. Picking the minimum of
> eleven noisy measurements gives you a number biased low: you have selected for
> lucky folds as much as for a good model. The cost is not a bad deliverable, it
> is seven experiments spent buying a difference you cannot measure.

## 2. What counts as an improvement

The tempting rule is "stop when the gain is under 0.001". That needs a different
number for every metric and every dataset, and getting it wrong is silent — too
strict and you stop mid-climb, too loose and you never stop.

The yardstick is already in the leaderboard. `fold_spread` is the standard
deviation of the incumbent's score across folds: how much this exact model's
number moves when you merely change which rows it trained on. A gain smaller
than that is not a smaller gain, it is the same measurement taken twice.

So: **a gain counts if it exceeds `min_gain_ratio` x the incumbent's fold
spread.** Everything else is a tie.

In [ ]:
from tabular_agent.harness.stop import StopPolicy, evaluate, advisory

policy = StopPolicy()          # patience=4, min_gain_ratio=0.25, max_experiments=25
print(policy)

status = evaluate(observed, "logloss", policy)
print(json.dumps(status, indent=2))

`since_improvement: 7` — seven experiments since anything real happened, against
a patience of 4. This run should have stopped at e8.

Note what the reason string does *not* say: it does not say the model is bad or
the features are exhausted. It says the last seven experiments produced no
measurable improvement, which is a claim about the search, not about the data.

## 3. Patience, and why it is not 1

Stopping at the first non-improvement would be wrong, because a run **branches**.
The agent builds a feature table, tries something on top that turns out worse,
backs up, tries a different encoder. A dip is the search working.

In [ ]:
branching = [
    {"id": "e1", "model_type": "lgbm", "cv_score": 0.40, "fold_spread": 0.02},
    {"id": "e2", "model_type": "lgbm", "cv_score": 0.55, "fold_spread": 0.03},   # worse
    {"id": "e3", "model_type": "lgbm", "cv_score": 0.52, "fold_spread": 0.03},   # still worse
    {"id": "e4", "model_type": "lgbm", "cv_score": 0.31, "fold_spread": 0.02},   # the payoff
]
for k in ("since_improvement", "should_stop"):
    print(k, "=", evaluate(branching, "logloss", policy)[k])

Two bad experiments in a row, and the run survives to find the good one. This is
early stopping, with `patience` meaning exactly what it means in boosting — and
inheriting the same trade-off: too little and you stop on a plateau that had
something behind it, too much and you pay for experiments you already know are
going nowhere.

**Check yourself:** the incumbent in that example stays `0.40` through e2 and e3,
even though those are the most recent results. Why not compare each experiment to
the one before it?

> Because "better than the last thing I tried" is a ratchet that never stops. A
> run alternating `0.40, 0.55, 0.41, 0.56, 0.42` would report an improvement every
> other step while going nowhere. The incumbent is the *best so far*, so beating it
> means beating everything — which is what "progress" has to mean.

## 4. Two layers: told, then stopped

Now the design question, and it is the same one as `train_tabular_model` writing
its own leaderboard row.

Putting "stop when gains stall" in the system prompt is **unfalsifiable**. A model
that keeps going has not violated anything you can point to, and you would never
know the guidance failed — exactly like asking an agent to log its own
experiments, where a forgotten row and an experiment that never happened look
identical from outside.

So the harness owns it. But a bare refusal produces a confused agent — it calls
the tool, gets rejected, does not know why, tries again. Hence two layers:

- **Advisory**: every training result carries `since_improvement` and a note. A
  cooperative model reads it and finalizes on its own, never getting refused.
- **Structural**: past the patience budget, the harness stops accepting training
  calls at all.

Guidance for the cooperative case, structure for the rest.

In [ ]:
print("fresh best :", advisory(evaluate(observed[:4], "logloss", policy)))
print()
print("stalling   :", advisory(evaluate(observed[:6], "logloss", policy)))
print()
print("over       :", advisory(evaluate(observed, "logloss", policy)))

## 5. Watch it happen

A tiny dataset and a deliberately impatient policy, driven by hand.

In [ ]:
import os, tempfile
import numpy as np
from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import inspect_data, train_tabular_model, finalize_run
from tabular_agent.harness import journal
from tabular_agent.taskspec import TaskSpec

rng = np.random.default_rng(0)
n = 300
x1 = rng.normal(size=n)
frame = pd.DataFrame({"x1": x1, "y": (x1 + rng.normal(scale=0.4, size=n) > 0).astype(int)})
workdir = tempfile.mkdtemp(prefix="phase3_step4_")
data_path = os.path.join(workdir, "d.csv")
frame.to_csv(data_path, index=False)

session = RunSession(
    run_dir=os.path.join(workdir, "run"),
    data_path=data_path,
    spec=TaskSpec(target="y", n_splits=3),
    stop_policy=StopPolicy(patience=2, max_refusals=2),
)
inspect_data(session)
train_tabular_model(session, model_type="baseline")

for i in range(3):
    out = train_tabular_model(session, model_type="lgbm")
    print(f"call {i + 1}: trained={out['trained']}", "|", out["note"][:90])

The same model on the same table three times, so the second and third cannot
possibly be improvements. All three still train — the check runs *before*
training, and when the third call arrives only one experiment had stalled. What
changes is the note: after the third, the advisory announces the run is over.

That ordering is worth keeping straight. The advisory describes the run *after*
the experiment you just did; the refusal is decided *before* the next one starts.
So a model always gets told once before it gets refused.

Now the fourth call. A refusal is a **return value**, not an exception — a real decision,
not a stylistic one. Raising would consume the tool's retry budget (`TOOL_RETRIES
= 3`), and exhausting that ends the run and throws away every completed
experiment. Being asked to train once more is not a malformed argument; it is a
model that has not yet read the news. It should cost nothing.

In [ ]:
refused = train_tabular_model(session, model_type="lgbm")
print(json.dumps({k: v for k, v in refused.items() if k != "note"}, indent=2)[:600])
print()
print("leaderboard is untouched:", [r["id"] for r in journal.load(session.run_dir)["leaderboard"]])

## 6. What if it will not take no for an answer

A model that keeps calling `train_tabular_model` after being refused would loop
until the context died — and the run would end with no bundle, which is the one
outcome that makes the whole run worthless.

So the harness escalates: after `max_refusals`, it packages the run **itself**
and hands back the finalize summary. Only once the deliverable is safely on disk
does it start raising.

In [ ]:
forced = train_tabular_model(session, model_type="lgbm")
print("harness_finalized:", forced.get("harness_finalized"))
print("bundle           :", forced.get("bundle_dir"))
print("note             :", forced.get("note"))

In [ ]:
try:
    train_tabular_model(session, model_type="lgbm")
except RuntimeError as e:
    print("RuntimeError:", e)

Read the escalation as three different judgements about the same call:

| Call | Response | Why |
|---|---|---|
| 1st after stalling | refusal + explanation | it just needs telling |
| `max_refusals`th | harness finalizes, returns the bundle | the deliverable matters more than being obeyed |
| after that | `RuntimeError` | the run is over; there is nothing left to protect |

**Check yourself:** `finalize_run` auto-trains a baseline if the run never did.
That is a call to `train_tabular_model`, made *by the harness*, while the run is
stalled. Why does the stop rule not refuse it?

> Because it is the harness training on its own behalf, not the agent asking for
> one more shot — hence `enforce_stop=False` on that call. Refusing it would
> leave the report unable to answer "did this beat doing nothing", which is the
> one question it must always be able to answer. Worth noticing that the rule had
> to be given an exception the moment it met the *other* rule; that is normal, and
> it is why the exception is a named argument with a comment rather than a
> coincidence.

## 7. Did it cost anything?

The step-3 run, uncapped, did 11 experiments and finished at `0.2718`. The same
prompt and model with the stop rule in place:

| | experiments | best | refusals |
|---|---|---|---|
| uncapped (step 3) | 11 | 0.2718 | — |
| with the stop rule | **8** | 0.2743 | 1 |

Three fewer experiments for a difference of `0.0025` — against a fold spread of
`0.0209`. In other words, it gave up nothing measurable, and the agent complied
with the first refusal rather than fighting it.

**Check yourself:** the capped run scored slightly *worse*. Is that a cost?

> Not one you can defend. `0.0025` is an eighth of the fold spread — the two
> numbers are the same measurement. If you treat that gap as real you have
> re-adopted the exact belief the stop rule exists to reject, and you will always
> be able to justify one more experiment.

## 8. What this does not do

Worth being precise about, because a stop rule is easy to over-read:

- It does **not** know whether the *features* are exhausted. It measures whether
  the search is still moving, and a run that keeps tuning hyper-parameters when it
  should be building features stalls for the right reason with the wrong lesson
  available. The advisory nudges ("changing the features is more likely to move
  the score"), but nudging is all it is.
- It does **not** protect against a bad metric or a leaky feature. A run leaking
  the target will converge fast and stop early, pleased with itself.
- `max_experiments` is a **budget**, not a judgement. A run that hits it may have
  been mid-climb, and the report will not say so.

**Next:** `blend_models` and `predict` are the two tools still missing, then §19
compaction — the run journal grows without bound, and a long run will eventually
outlive the context it is being read into.